# 07 Branch-and-Price —— 节点 RMP 的对偶、分支与剪枝的数学

## 节点 RMP 与对偶

每个 B&B 节点解（含车辆数上下界与虚拟列 $y_i$，成本 $M$）：

$$\min\ \sum_p c_px_p+M\sum_i y_i \quad\text{s.t.}\quad \sum_p a_{ip}x_p+y_i\ge 1\ (\pi_i\ge 0),\ \sum_p x_p\le K_{ub}\ (\mu_{ub}\le 0),\ \sum_p x_p\ge K_{lb}\ (\mu_{lb}\ge 0)$$

按 02 的对偶规则：**下界约束（≥）对偶 $\mu_{lb}\ge 0$**，上界约束（≤）对偶 $\mu_{ub}\le 0$，虚拟列给 $\pi_i\le M$：

$$\max\ \sum_i\pi_i+K_{ub}\mu_{ub}+K_{lb}\mu_{lb}\ \ \text{s.t.}\ \ \sum_i a_{ip}\pi_i+\mu_{ub}+\mu_{lb}\le c_p\ (\forall p),\ \pi_i\le M,\ \pi_i\ge 0,\ \mu_{ub}\le 0,\ \mu_{lb}\ge 0$$

**定价 reduced cost**（两车辆对偶合并）：
$$rc_p=c_p-\sum_{i\in p}\pi_i-(\mu_{ub}+\mu_{lb})$$

- 定价子问题：02 的 CP-SAT ESPPRC + **强制弧 $x_{ij}=1$ / 禁用弧 $x_{ij}=0$**（分支约束传播进定价）；
  池扫描同步按弧过滤。
- 虚拟列取正值 ⇒ 节点无真实覆盖 ⇒ 不可行剪枝（$\pi_i=M$ 时所有真实列 rc<0 但池中无符合弧限制的列）。

## 分支规则的数学

- **车辆数分支**：$\Sigma_p x_p=f$ 分数 → 左子 $\Sigma x\le\lfloor f\rfloor$、右子 $\Sigma x\ge\lceil f\rceil$，
  把分数车辆数区间剖开（对应 RMP 的 $K_{ub}/K_{lb}$）。
- **Ryan-Foster 弧分支**：定义弧流量 $f_{ij}=\sum_p x_p\cdot\mathbb 1[(i,j)\in p]$。
  **事实**：$x$ 分数 ⇒ 存在客户弧 $(i,j)$ 使 $f_{ij}\in(0,1)$（若全部 $f_{ij}\in\{0,1\}$，由流量守恒可证各列 $x_p\in\{0,1\}$）。
  取最接近 0.5 的弧分支：左子禁用（$x_{ij}=0$）、右子强制（$x_{ij}=1$），子代弧集不相交 ⇒ 树有限。

## 剪枝正确性（三个引理）

1. **定界剪枝**：节点 LP 值 ≥ 现任解 UB ⇒ 子树不可能出现更优整数解（节点 LP 是其下界）。
2. **不可行剪枝**：虚拟列取正 ⇒ 不存在满足分支弧限制的真实覆盖。
3. **整数节点**：$x_p\in\{0,1\}$ ⇒ LP 解即整数解，更新 UB 后剪枝（节点已解决）。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


# 节点演示：K_lb=K_ub=3 的节点 RMP（28 列池 + 虚拟列）
M = 10**6
K_lb = K_ub = 3
m = mathopt.Model()
xv = [m.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
yv = [m.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"y{i}") for i in range(1, n+1)]
covers = []
for i in range(1, n+1):
    covers.append(m.add_linear_constraint(
        mathopt.fast_sum([xv[p] for p in range(P) if (masks[p] >> i) & 1]) + yv[i-1] >= 1.0, name=f"c{i}"))
ubc = m.add_linear_constraint(mathopt.fast_sum(xv) <= K_ub, name="vub")
lbc = m.add_linear_constraint(mathopt.fast_sum(xv) >= K_lb, name="vlb")
m.minimize(mathopt.fast_sum([costs[p]*xv[p] for p in range(P)]) + M*mathopt.fast_sum(yv))
res = mathopt.solve(m, mathopt.SolverType.GLOP)
dv = res.dual_values()
pi = [0.0]*(n+1)
for i in range(1, n+1):
    pi[i] = max(0.0, dv[covers[i-1]])
mu_ub = dv[ubc]
mu_lb = dv[lbc]
xvals = {p: res.variable_values()[xv[p]] for p in range(P)}
y_used = sum(res.variable_values()[yv[i-1]] for i in range(1, n+1))
print("节点 LP =", round(res.objective_value(), 6), "| 虚拟列使用 =", round(y_used, 6))
print("mu_ub =", round(mu_ub, 4), "| mu_lb =", round(mu_lb, 4),
      "| rc = c - Σπ - μ_ub - μ_lb（正列验证）:")
for p in range(P):
    if xvals[p] > 1e-6:
        s = 0.0
        mm = masks[p]
        while mm:
            lb = mm & -mm
            s += pi[lb.bit_length()-1]
            mm -= lb
        rc = costs[p] - s - mu_ub - mu_lb
        print(f"  列 {pool[p]}  x={round(xvals[p],3)}  rc={rc:.2e}")
# 弧流量与整数性
flows = {}
for p, val in xvals.items():
    if val <= 1e-6:
        continue
    seq = [0] + list(pool[p]) + [0]
    for a in range(len(seq)-1):
        flows[(seq[a], seq[a+1])] = flows.get((seq[a], seq[a+1]), 0.0) + val
frac_arcs = [(a, f) for a, f in flows.items() if a[0] >= 1 and a[1] >= 1 and 1e-6 < f < 1-1e-6]
integral = all(abs(v-round(v)) < 1e-6 for v in xvals.values()) and y_used <= 1e-6
print("分数客户弧数:", len(frac_arcs), "| LP 整数解:", integral,
      "-> 根节点即证明最优（无需分支）" if integral else "-> 需分支（车辆数/弧）")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）
节点 LP = 191.81362 | 虚拟列使用 = 0.0
mu_ub = -5999963.5593 | mu_lb = 0.0 | rc = c - Σπ - μ_ub - μ_lb（正列验证）:
  列 (13, 17, 18, 19, 15, 16, 14, 12)  x=1.0  rc=9.31e-10
  列 (20, 24, 25, 23, 22, 21)  x=1.0  rc=-9.31e-10
  列 (5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1)  x=1.0  rc=0.00e+00
分数客户弧数: 0 | LP 整数解: True -> 根节点即证明最优（无需分支）
